# Classificação de Qualidade da Água com XGBoost

Pipeline de **classificação binária** da potabilidade da água superficial com base em oito parâmetros físico-químicos, usando **XGBoost** e otimização de hiperparâmetros com **Optuna**.

## Dataset

- **Arquivo:** `Combined_dataset.csv` (Karim et al., 2025)
- **Registros:** ~2,82 milhões de medições (1940–2023)
- **Target original:** `CCME_WQI` (índice CCME)
- **Target binário:** Potável (1) vs Não Potável (0)

## Objetivo

Prever se a água é **potável** (`Excellent`, `Good`) ou **não potável** (`Fair`, `Marginal`, `Poor`) a partir das features físico-químicas, **sem** usar variáveis derivadas do próprio índice (evitar data leakage).

## Referências iniciais

- Karim et al. (2025) — dataset de qualidade da água superficial
- Santos & Polo (2025) — XGBoost aplicado à qualidade da água
- Chen & Guestrin (2016) — XGBoost
- Lai et al. (2023) — modelos tree-based com Optuna

In [ ]:
# Stdlib
import json
import logging
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

# Terceiros
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBClassifier

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)
optuna.logging.set_verbosity(optuna.logging.WARNING)


: 

In [ ]:
DATA_PATH = Path("data/Combined_dataset.csv")
OUTPUT_DIR = Path("outputs")
REPORTS_DIR = OUTPUT_DIR / "reports"
PLOTS_DIR = OUTPUT_DIR / "plots"
MODELS_DIR = OUTPUT_DIR / "models"

FEATURE_COLUMNS = [
    "Ammonia (mg/l)",
    "Biochemical Oxygen Demand (mg/l)",
    "Dissolved Oxygen (mg/l)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Temperature (cel)",
    "Nitrogen (mg/l)",
    "Nitrate (mg/l)",
]
TARGET_COLUMN = "CCME_WQI"
POTABLE_CLASSES = ["Excellent", "Good"]
NON_POTABLE_CLASSES = ["Fair", "Marginal", "Poor"]
BINARY_TARGET_COLUMN = "potable"

CHUNK_SIZE = 100_000
TEST_SIZE = 0.20
RANDOM_STATE = 42
CV_FOLDS = 7
OPTUNA_TRIALS = 150
OPTUNA_SAMPLE_SIZE = 300_000
EDA_SAMPLE_SIZE = 100_000
IQR_MULTIPLIER = 1.5
CLASSIFICATION_THRESHOLD = 0.5

TARGET_ACCURACY = 0.90
TARGET_AUC_ROC = 0.90
TARGET_RECALL_NON_POTABLE = 0.85

EDA_SAMPLE_LABEL = "(amostra de 100.000 registros)"

for directory in (REPORTS_DIR, PLOTS_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)


## Seção 1: Carregamento dos Dados

Carrega o CSV em chunks, seleciona features e target, cria a variável binária `potable` e registra um resumo do dataset.

In [ ]:
class DataLoader:
    """Carrega e prepara o dataset de qualidade da água."""

    def load_dataset(self, file_path: Path, chunksize: int) -> pd.DataFrame:
        """Carrega o CSV em chunks para evitar estouro de memória."""
        if not file_path.exists():
            raise FileNotFoundError(f"Arquivo não encontrado: {file_path}")
        columns = FEATURE_COLUMNS + [TARGET_COLUMN]
        chunks: list[pd.DataFrame] = []
        try:
            reader = pd.read_csv(file_path, usecols=columns, chunksize=chunksize)
            for chunk_index, chunk in enumerate(reader, start=1):
                chunks.append(chunk)
                logger.info("Chunk %s carregado (%s linhas).", chunk_index, len(chunk))
        except (pd.errors.ParserError, OSError) as error:
            logger.exception("Falha ao ler o dataset.")
            raise RuntimeError(f"Erro ao carregar {file_path}: {error}") from error
        return pd.concat(chunks, ignore_index=True)

    def binarize_target(self, target_series: pd.Series) -> pd.Series:
        """Converte CCME_WQI em rótulo binário potável (1) / não potável (0)."""
        return target_series.isin(POTABLE_CLASSES).astype(int)

    def log_summary(self, dataframe: pd.DataFrame) -> None:
        """Registra shape, distribuição do target, nulos e estatísticas."""
        logger.info("Shape do dataset: %s", dataframe.shape)
        logger.info("Distribuição do target:\n%s", dataframe[BINARY_TARGET_COLUMN].value_counts())
        logger.info("Nulos por coluna:\n%s", dataframe.isnull().sum())
        logger.info("Describe:\n%s", dataframe[FEATURE_COLUMNS].describe())
        summary = {
            "shape": list(dataframe.shape),
            "target_distribution": dataframe[BINARY_TARGET_COLUMN].value_counts().to_dict(),
            "null_counts": dataframe.isnull().sum().to_dict(),
            "describe": dataframe[FEATURE_COLUMNS].describe().to_dict(),
        }
        summary_path = REPORTS_DIR / "dataset_summary.json"
        with summary_path.open("w", encoding="utf-8") as file:
            json.dump(summary, file, indent=2)
        logger.info("Resumo salvo em %s", summary_path)


In [ ]:
data_loader = DataLoader()
raw_dataframe = data_loader.load_dataset(DATA_PATH, CHUNK_SIZE)
raw_dataframe[BINARY_TARGET_COLUMN] = data_loader.binarize_target(raw_dataframe[TARGET_COLUMN])
# CCME_Values/CCME_WQI e metadados (Country, Date, etc.) não entram como feature: leakage ou sem valor físico-químico.
raw_dataframe = raw_dataframe[FEATURE_COLUMNS + [BINARY_TARGET_COLUMN]]
data_loader.log_summary(raw_dataframe)
raw_dataframe.head()


## Seção 2: Análise Exploratória (EDA)

Visualizações sobre amostra aleatória de 100.000 registros para manter o kernel responsivo com ~2,82 M de linhas.

In [ ]:
def create_eda_sample(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Extrai amostra aleatória de tamanho fixo para a EDA."""
    return dataframe.sample(n=EDA_SAMPLE_SIZE, random_state=RANDOM_STATE)


def plot_target_distribution(eda_sample: pd.DataFrame) -> None:
    """Gera countplot da distribuição do target binário."""
    figure, axis = plt.subplots(figsize=(8, 5))
    sns.countplot(data=eda_sample, x=BINARY_TARGET_COLUMN, ax=axis)
    axis.set_title(f"Distribuição do target potável {EDA_SAMPLE_LABEL}")
    axis.set_xlabel("Potável (1) / Não Potável (0)")
    figure.tight_layout()
    figure.savefig(PLOTS_DIR / "target_distribution.png", dpi=150)
    plt.show()


def plot_feature_boxplots(eda_sample: pd.DataFrame) -> None:
    """Boxplots das features separados por classe binária."""
    melted = eda_sample.melt(
        id_vars=BINARY_TARGET_COLUMN,
        value_vars=FEATURE_COLUMNS,
        var_name="feature",
        value_name="value",
    )
    figure, axis = plt.subplots(figsize=(14, 8))
    sns.boxplot(
        data=melted,
        x="feature",
        y="value",
        hue=BINARY_TARGET_COLUMN,
        ax=axis,
    )
    axis.set_title(f"Boxplots por feature e classe {EDA_SAMPLE_LABEL}")
    axis.tick_params(axis="x", rotation=45)
    figure.tight_layout()
    figure.savefig(PLOTS_DIR / "feature_boxplots.png", dpi=150)
    plt.show()


def plot_correlation_heatmap(eda_sample: pd.DataFrame) -> None:
    """Heatmap de correlação entre as oito features."""
    correlation = eda_sample[FEATURE_COLUMNS].corr()
    figure, axis = plt.subplots(figsize=(10, 8))
    sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", ax=axis)
    axis.set_title(f"Correlação entre features {EDA_SAMPLE_LABEL}")
    figure.tight_layout()
    plt.show()


def plot_feature_histograms(eda_sample: pd.DataFrame) -> None:
    """Histogramas da distribuição de cada feature."""
    figure, axes = plt.subplots(2, 4, figsize=(16, 8))
    for axis, column in zip(axes.flatten(), FEATURE_COLUMNS):
        sns.histplot(eda_sample[column], kde=True, ax=axis)
        axis.set_title(column)
    figure.suptitle(f"Distribuição das features {EDA_SAMPLE_LABEL}", y=1.02)
    figure.tight_layout()
    plt.show()


eda_sample = create_eda_sample(raw_dataframe)
plot_target_distribution(eda_sample)
plot_feature_boxplots(eda_sample)
plot_correlation_heatmap(eda_sample)
plot_feature_histograms(eda_sample)


## Seção 3: Pré-processamento

Split estratificado, clipping IQR (fit no treino) e MinMaxScaler (fit no treino). Sem imputação, SMOTE ou balanceamento artificial.

In [ ]:
class DataPreprocessor:
    """Aplica clipping IQR e normalização MinMax."""

    def __init__(self) -> None:
        self._iqr_bounds: dict[str, tuple[float, float]] = {}
        self._scaler = MinMaxScaler()

    def fit_transform(self, features_train: pd.DataFrame) -> pd.DataFrame:
        """Ajusta transformações no treino e retorna features processadas."""
        self._iqr_bounds = self._compute_iqr_bounds(features_train)
        clipped = self._clip_outliers(features_train)
        return self._scale_features(clipped, fit=True)

    def transform(self, features: pd.DataFrame) -> pd.DataFrame:
        """Aplica transformações já ajustadas em novos dados."""
        clipped = self._clip_outliers(features)
        return self._scale_features(clipped, fit=False)

    def _compute_iqr_bounds(self, features: pd.DataFrame) -> dict[str, tuple[float, float]]:
        """Calcula limites inferior e superior por coluna via IQR."""
        bounds: dict[str, tuple[float, float]] = {}
        for column in features.columns:
            q1 = features[column].quantile(0.25)
            q3 = features[column].quantile(0.75)
            iqr = q3 - q1
            lower = q1 - IQR_MULTIPLIER * iqr
            upper = q3 + IQR_MULTIPLIER * iqr
            bounds[column] = (lower, upper)
        return bounds

    def _clip_outliers(self, features: pd.DataFrame) -> pd.DataFrame:
        """Limita valores extremos aos bounds IQR calculados no treino."""
        clipped = features.copy()
        for column, (lower, upper) in self._iqr_bounds.items():
            clipped[column] = clipped[column].clip(lower=lower, upper=upper)
        return clipped

    def _scale_features(self, features: pd.DataFrame, fit: bool) -> pd.DataFrame:
        """Escala features para [0, 1] com MinMaxScaler."""
        if fit:
            scaled = self._scaler.fit_transform(features)
        else:
            scaled = self._scaler.transform(features)
        return pd.DataFrame(scaled, columns=features.columns, index=features.index)


In [ ]:
feature_matrix = raw_dataframe[FEATURE_COLUMNS]
target_vector = raw_dataframe[BINARY_TARGET_COLUMN]

features_train, features_test, target_train, target_test = train_test_split(
    feature_matrix,
    target_vector,
    test_size=TEST_SIZE,
    stratify=target_vector,
    random_state=RANDOM_STATE,
)

data_preprocessor = DataPreprocessor()
features_train_scaled = data_preprocessor.fit_transform(features_train)
features_test_scaled = data_preprocessor.transform(features_test)

logger.info("Treino: %s | Teste: %s", features_train_scaled.shape, features_test_scaled.shape)


## Seção 4: Treinamento e Otimização

Otimização com Optuna (150 trials, CV estratificado k=7) sobre amostra de 300.000 registros do treino; modelo final no treino completo.

In [ ]:
class ModelTrainer:
    """Otimiza hiperparâmetros com Optuna e treina XGBClassifier."""

    def optimize_hyperparameters(self, features: pd.DataFrame, target: pd.Series) -> dict[str, Any]:
        """Executa busca bayesiana maximizando acurácia em CV estratificado."""
        sample_features, sample_target = self._build_optuna_sample(features, target)
        study = optuna.create_study(direction="maximize")
        study.optimize(
            lambda trial: self._objective(trial, sample_features, sample_target),
            n_trials=OPTUNA_TRIALS,
            show_progress_bar=True,
        )
        logger.info("Melhor acurácia CV: %.4f", study.best_value)
        logger.info("Melhores hiperparâmetros: %s", study.best_params)
        return study.best_params

    def train(self, features: pd.DataFrame, target: pd.Series, params: dict[str, Any]) -> XGBClassifier:
        """Treina o modelo final com os hiperparâmetros informados."""
        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=-1,
            random_state=RANDOM_STATE,
            tree_method="hist",
            **params,
        )
        model.fit(features, target)
        model_path = MODELS_DIR / "xgb_water_quality.json"
        model.save_model(model_path)
        logger.info("Modelo salvo em %s", model_path)
        return model

    def _build_optuna_sample(
        self, features: pd.DataFrame, target: pd.Series
    ) -> tuple[pd.DataFrame, pd.Series]:
        """Extrai amostra estratificada de 300k para o Optuna manter proporção de classes."""
        sample_features, _, sample_target, _ = train_test_split(
            features,
            target,
            train_size=OPTUNA_SAMPLE_SIZE,
            stratify=target,
            random_state=RANDOM_STATE,
        )
        return sample_features, sample_target

    def _suggest_hyperparameters(self, trial: optuna.Trial) -> dict[str, Any]:
        """Define o espaço de busca do Optuna para um trial."""
        return {
            "n_estimators": trial.suggest_int("n_estimators", 50, 500),
            "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True),
            "max_depth": trial.suggest_int("max_depth", 1, 15),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }

    def _objective(self, trial: optuna.Trial, features: pd.DataFrame, target: pd.Series) -> float:
        """Função objetivo do Optuna: acurácia média em StratifiedKFold."""
        params = self._suggest_hyperparameters(trial)
        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=-1,
            random_state=RANDOM_STATE,
            tree_method="hist",
            **params,
        )
        cross_validator = StratifiedKFold(
            n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE
        )
        scores = cross_val_score(
            model, features, target, cv=cross_validator, scoring="accuracy", n_jobs=-1
        )
        return float(scores.mean())


In [ ]:
model_trainer = ModelTrainer()
best_hyperparameters = model_trainer.optimize_hyperparameters(features_train_scaled, target_train)
trained_model = model_trainer.train(features_train_scaled, target_train, best_hyperparameters)


## Seção 5: Avaliação

Métricas no conjunto de teste, gráficos salvos em `outputs/plots/` e relatórios em `outputs/reports/`.

In [ ]:
@dataclass
class EvaluationReport:
    """Resultados da avaliação no conjunto de teste."""

    accuracy: float
    precision_per_class: dict[int, float]
    recall_per_class: dict[int, float]
    f1_per_class: dict[int, float]
    f1_macro: float
    auc_roc: float
    confusion: list[list[int]]
    classification_report_text: str
    compliance: dict[str, str] = field(default_factory=dict)


class Evaluator:
    """Avalia o modelo, gera gráficos e salva artefatos."""

    def __init__(self, model: XGBClassifier, preprocessor: DataPreprocessor) -> None:
        self._model = model
        self._preprocessor = preprocessor

    def evaluate(
        self,
        features_test: pd.DataFrame,
        target_test: pd.Series,
    ) -> EvaluationReport:
        """Calcula métricas e gera visualizações no conjunto de teste."""
        probabilities = self._model.predict_proba(features_test)[:, 1]
        predictions = (probabilities >= CLASSIFICATION_THRESHOLD).astype(int)
        report = self._build_report(target_test, predictions, probabilities)
        self._log_target_compliance(report)
        self._plot_confusion_matrix(target_test, predictions)
        self._plot_roc_curve(target_test, probabilities)
        self._plot_feature_importance()
        self.save_artifacts(report)
        return report

    def save_artifacts(self, report: EvaluationReport) -> None:
        """Persiste relatórios em texto e JSON."""
        text_path = REPORTS_DIR / "evaluation_report.txt"
        json_path = REPORTS_DIR / "evaluation_report.json"
        text_content = self._format_text_report(report)
        with text_path.open("w", encoding="utf-8") as file:
            file.write(text_content)
        with json_path.open("w", encoding="utf-8") as file:
            json.dump(asdict(report), file, indent=2)
        logger.info("Relatórios salvos em %s e %s", text_path, json_path)

    def _build_report(
        self,
        target_test: pd.Series,
        predictions: np.ndarray,
        probabilities: np.ndarray,
    ) -> EvaluationReport:
        """Monta o objeto EvaluationReport com todas as métricas."""
        labels = [0, 1]
        return EvaluationReport(
            accuracy=float(accuracy_score(target_test, predictions)),
            precision_per_class={
                label: float(precision_score(target_test, predictions, pos_label=label, zero_division=0))
                for label in labels
            },
            recall_per_class={
                label: float(recall_score(target_test, predictions, pos_label=label, zero_division=0))
                for label in labels
            },
            f1_per_class={
                label: float(f1_score(target_test, predictions, pos_label=label, zero_division=0))
                for label in labels
            },
            f1_macro=float(f1_score(target_test, predictions, average="macro")),
            auc_roc=float(roc_auc_score(target_test, probabilities)),
            confusion=confusion_matrix(target_test, predictions).tolist(),
            classification_report_text=classification_report(target_test, predictions),
        )

    def _format_text_report(self, report: EvaluationReport) -> str:
        """Formata relatório textual para arquivo."""
        lines = [
            f"Accuracy: {report.accuracy:.4f}",
            f"AUC-ROC: {report.auc_roc:.4f}",
            f"F1 Macro: {report.f1_macro:.4f}",
            f"Precision (0/1): {report.precision_per_class}",
            f"Recall (0/1): {report.recall_per_class}",
            f"F1 (0/1): {report.f1_per_class}",
            "",
            report.classification_report_text,
            "",
            "Metas:",
        ]
        for metric, status in report.compliance.items():
            lines.append(f"  {metric}: {status}")
        return chr(10).join(lines)

    def _plot_confusion_matrix(self, target_test: pd.Series, predictions: np.ndarray) -> None:
        """Salva e exibe heatmap da matriz de confusão."""
        matrix = confusion_matrix(target_test, predictions)
        figure, axis = plt.subplots(figsize=(6, 5))
        sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", ax=axis)
        axis.set_xlabel("Predito")
        axis.set_ylabel("Real")
        axis.set_title("Matriz de Confusão")
        figure.tight_layout()
        figure.savefig(PLOTS_DIR / "confusion_matrix.png", dpi=150)
        plt.show()

    def _plot_roc_curve(self, target_test: pd.Series, probabilities: np.ndarray) -> None:
        """Salva e exibe curva ROC com AUC."""
        false_positive, true_positive, _ = roc_curve(target_test, probabilities)
        roc_auc = auc(false_positive, true_positive)
        figure, axis = plt.subplots(figsize=(7, 5))
        axis.plot(false_positive, true_positive, label=f"AUC = {roc_auc:.4f}")
        axis.plot([0, 1], [0, 1], linestyle="--", color="gray")
        axis.set_xlabel("FPR")
        axis.set_ylabel("TPR")
        axis.set_title("Curva ROC")
        axis.legend()
        figure.tight_layout()
        figure.savefig(PLOTS_DIR / "roc_curve.png", dpi=150)
        plt.show()

    def _plot_feature_importance(self) -> None:
        """Salva e exibe importância das features (barh)."""
        importance = self._model.feature_importances_
        figure, axis = plt.subplots(figsize=(8, 6))
        axis.barh(FEATURE_COLUMNS, importance)
        axis.set_title("Feature Importance (XGBoost)")
        axis.invert_yaxis()
        figure.tight_layout()
        figure.savefig(PLOTS_DIR / "feature_importance.png", dpi=150)
        plt.show()

    def _log_target_compliance(self, report: EvaluationReport) -> None:
        """Registra se as metas de acurácia, AUC e recall foram atingidas."""
        report.compliance = {
            "Acurácia >= 0.90": "OK" if report.accuracy >= TARGET_ACCURACY else "NAO ATINGIDO",
            "AUC-ROC >= 0.90": "OK" if report.auc_roc >= TARGET_AUC_ROC else "NAO ATINGIDO",
            "Recall Não Potável >= 0.85": (
                "OK" if report.recall_per_class[0] >= TARGET_RECALL_NON_POTABLE else "NAO ATINGIDO"
            ),
        }
        for metric, status in report.compliance.items():
            logger.info("%s → %s", metric, status)


In [ ]:
evaluator = Evaluator(trained_model, data_preprocessor)
evaluation_report = evaluator.evaluate(features_test_scaled, target_test)
evaluation_report


## Teste manual do modelo

Amostra **10 registros não potáveis** e **10 potáveis** do conjunto de teste (features brutas, antes da escala). O pré-processamento é reaplicado com o `DataPreprocessor` já ajustado no treino; as predições são comparadas com a classe real.

In [28]:
from IPython.display import display

MANUAL_TEST_SAMPLE_COUNT = 10
CLASS_LABEL_NAMES = {0: "Não Potável", 1: "Potável"}


def sample_test_by_class(
    features: pd.DataFrame,
    target: pd.Series,
    class_label: int,
    sample_count: int,
) -> tuple[pd.DataFrame, pd.Series]:
    """Seleciona amostras aleatórias de uma classe no conjunto de teste."""
    class_mask = target == class_label
    class_features = features.loc[class_mask]
    class_target = target.loc[class_mask]
    sample_size = min(sample_count, len(class_features))
    sampled = class_features.sample(n=sample_size, random_state=RANDOM_STATE)
    return sampled, class_target.loc[sampled.index]


def build_prediction_table(
    model: XGBClassifier,
    preprocessor: DataPreprocessor,
    raw_features: pd.DataFrame,
    true_labels: pd.Series,
) -> pd.DataFrame:
    """Aplica pré-processamento, prediz e monta tabela com classe real vs predita."""
    scaled_features = preprocessor.transform(raw_features)
    probabilities = model.predict_proba(scaled_features)[:, 1]
    predictions = (probabilities >= CLASSIFICATION_THRESHOLD).astype(int)
    results = raw_features.copy()
    results["classe_real"] = true_labels.map(CLASS_LABEL_NAMES)
    results["classe_predita"] = pd.Series(predictions, index=raw_features.index).map(
        CLASS_LABEL_NAMES
    )
    results["prob_potavel"] = np.round(probabilities, 4)
    results["acertou"] = true_labels.to_numpy() == predictions
    return results


def log_manual_test_summary(group_name: str, results: pd.DataFrame) -> None:
    """Registra quantos acertos houve nas amostras manuais."""
    hits = int(results["acertou"].sum())
    total = len(results)
    accuracy_pct = 100.0 * hits / total if total else 0.0
    logger.info("%s: %s/%s acertos (%.1f%%)", group_name, hits, total, accuracy_pct)


non_potable_features, non_potable_labels = sample_test_by_class(
    features_test, target_test, class_label=0, sample_count=MANUAL_TEST_SAMPLE_COUNT
)
potable_features, potable_labels = sample_test_by_class(
    features_test, target_test, class_label=1, sample_count=MANUAL_TEST_SAMPLE_COUNT
)

non_potable_results = build_prediction_table(
    trained_model, data_preprocessor, non_potable_features, non_potable_labels
)
potable_results = build_prediction_table(
    trained_model, data_preprocessor, potable_features, potable_labels
)

log_manual_test_summary("Não Potável (10 amostras)", non_potable_results)
log_manual_test_summary("Potável (10 amostras)", potable_results)

logger.info("--- Amostras Não Potáveis (classe real = 0) ---")
display(non_potable_results)

logger.info("--- Amostras Potáveis (classe real = 1) ---")
display(potable_results)

2026-05-19 00:08:25 | INFO | Não Potável (10 amostras): 9/10 acertos (90.0%)
2026-05-19 00:08:25 | INFO | Potável (10 amostras): 10/10 acertos (100.0%)
2026-05-19 00:08:25 | INFO | --- Amostras Não Potáveis (classe real = 0) ---


,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),classe_real,classe_predita,prob_potavel,acertou
1067684,1.340,2.92,10.20,6.120,7.95,13.36,20.80,4.50,Não Potável,Não Potável,0.0020,True
1952080,1.890,2.70,11.80,0.186,7.87,2.62,5.24,5.16,Não Potável,Potável,0.6813,False
643593,7.220,19.70,9.43,0.144,7.20,16.08,5.00,4.50,Não Potável,Não Potável,0.0800,True
1751481,10.000,11.00,10.20,0.144,7.60,11.46,5.00,1.10,Não Potável,Não Potável,0.0085,True
1076764,0.500,2.85,10.20,2.880,7.10,10.70,12.20,4.50,Não Potável,Não Potável,0.0021,True
380364,0.230,2.70,8.60,0.460,7.43,14.90,16.00,16.00,Não Potável,Não Potável,0.0027,True
360356,0.750,13.00,8.23,5.000,7.57,12.20,43.00,42.70,Não Potável,Não Potável,0.0021,True
1407729,2.340,6.71,10.20,6.150,7.15,11.08,25.80,4.50,Não Potável,Não Potável,0.0018,True
1469632,3.400,6.35,10.20,0.144,6.92,11.46,5.00,4.50,Não Potável,Não Potável,0.0724,True
489589,0.238,2.70,9.10,0.500,7.70,9.70,3.28,4.50,Não Potável,Não Potável,0.0051,True


2026-05-19 00:08:25 | INFO | --- Amostras Potáveis (classe real = 1) ---


,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),classe_real,classe_predita,prob_potavel,acertou
2819022,0.067,1.4,4.7025,0.025,8.30,15.60,1.90,1.900,Potável,Potável,0.9986,True
1571902,0.030,2.7,10.2000,0.083,7.85,11.46,5.00,4.500,Potável,Potável,0.9990,True
2448779,0.021,1.6,1.1400,0.040,7.00,11.10,0.40,0.900,Potável,Potável,0.9992,True
1832242,0.094,2.7,13.1000,0.026,8.19,3.97,4.98,4.960,Potável,Potável,0.9987,True
205413,0.030,2.7,12.1000,0.010,7.73,8.10,0.20,0.196,Potável,Potável,0.9990,True
1771088,0.070,2.0,11.0000,0.237,7.53,13.00,1.16,1.140,Potável,Potável,0.9981,True
2008921,0.030,2.7,7.7800,0.023,8.35,10.40,2.04,2.020,Potável,Potável,0.9989,True
338173,0.200,6.0,10.2000,0.144,7.26,11.46,5.00,4.500,Potável,Potável,0.9803,True
1382614,0.098,2.7,9.3000,0.144,7.27,14.02,2.39,4.500,Potável,Potável,0.9980,True
1346116,0.515,5.1,10.2000,0.099,7.78,11.46,1.97,4.500,Potável,Potável,0.6458,True


## Orquestração do Pipeline

A classe `WaterQualityPipeline` existe para **reexecução programática** do fluxo completo em uma única chamada (`run()`).

> **Ao rodar o notebook célula a célula**, as seções 1–5 já executaram o pipeline passo a passo (carregamento, EDA, pré-processamento, treino e avaliação). **Não é necessário** chamar `WaterQualityPipeline().run()` em seguida — isso repetiría todo o processamento (~2,82 M de registros) do zero.
>
> Use `run()` apenas se quiser reexecutar o pipeline de ponta a ponta (por exemplo, em script externo ou após alterar constantes na célula de configuração).

In [ ]:
class WaterQualityPipeline:
    """Orquestra carregamento, pré-processamento, treino e avaliação."""

    def run(self) -> EvaluationReport:
        """Executa o pipeline de ponta a ponta e retorna o relatório de avaliação."""
        loader = DataLoader()
        dataframe = loader.load_dataset(DATA_PATH, CHUNK_SIZE)
        dataframe[BINARY_TARGET_COLUMN] = loader.binarize_target(dataframe[TARGET_COLUMN])
        dataframe = dataframe[FEATURE_COLUMNS + [BINARY_TARGET_COLUMN]]
        loader.log_summary(dataframe)

        features = dataframe[FEATURE_COLUMNS]
        target = dataframe[BINARY_TARGET_COLUMN]
        features_tr, features_te, target_tr, target_te = train_test_split(
            features, target, test_size=TEST_SIZE, stratify=target, random_state=RANDOM_STATE
        )

        preprocessor = DataPreprocessor()
        features_tr = preprocessor.fit_transform(features_tr)
        features_te = preprocessor.transform(features_te)

        trainer = ModelTrainer()
        params = trainer.optimize_hyperparameters(features_tr, target_tr)
        model = trainer.train(features_tr, target_tr, params)

        return Evaluator(model, preprocessor).evaluate(features_te, target_te)


# Descomente para reexecutar o pipeline completo em uma única chamada:
# evaluation_report = WaterQualityPipeline().run()


## Seção 6: Conclusão

### Resumo

Este notebook implementa um pipeline reprodutível de classificação binária da potabilidade da água com **XGBoost**, oito parâmetros físico-químicos e binarização do índice **CCME_WQI**. O pré-processamento usa **IQR clipping** e **MinMaxScaler** (ajuste apenas no treino), e a otimização usa **Optuna** com validação cruzada estratificada sobre uma amostra de 300.000 registros.

### Metas de desempenho

Verifique no log e em `outputs/reports/evaluation_report.txt` se foram atingidas:

- Acurácia ≥ 0,90
- AUC-ROC ≥ 0,90
- Recall da classe Não Potável (0) ≥ 0,85

### Limitações

- O modelo depende da qualidade e cobertura geográfica do dataset Karim et al. (2025).
- Features excluídas por **data leakage** (`CCME_Values`, `CCME_WQI`) não podem ser usadas em produção sem redefinir o problema.
- Optuna e EDA usam amostras por restrição computacional; o modelo final treina no treino completo (~80% dos dados).

### Trabalhos futuros

- Análise de calibração de probabilidades e custo assimétrico de erros.
- Explicabilidade local (SHAP) por região e tipo de corpo d'água.
- Validação temporal por `Date` quando disponível em cenários operacionais.